1. Objetivo

01 — Landing Zone Validation

Objetivo: validar ingestão de arquivos RAW para a Landing Zone no MinIO.

Valida:
- descoberta de arquivos locais
- cálculo de hash
- upload para MinIO
- preservação de snapshots
- geração de metadados

2. Setup

In [7]:
from src.landing.loader import run_landing_pipeline
from src.config.settings import Settings
import boto3

3. criar landing_metadata_df

In [8]:
landing_metadata_df = run_landing_pipeline()

landing_metadata_df["upload_status"].value_counts()

upload_status
SUCCESS    36
Name: count, dtype: int64

4. Visualizar metadados

In [9]:
landing_metadata_df[
    [
        "snapshot_date",
        "source_file",
        "file_extension",
        "file_size_bytes",
        "file_hash",
        "upload_status"
    ]
].head(10)

,snapshot_date,source_file,file_extension,file_size_bytes,file_hash,upload_status
0,2024-07-13_0800,ControleMedicoesPagamentos.csv,.csv,39634,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,SUCCESS
1,2024-07-13_0800,Exportação_bm_acompanhamento.xlsx,.xlsx,18912,84df6ac3cb58f7ca789bf83e192ad99369bb1d35c6b41f...,SUCCESS
2,2024-07-13_0800,202211_ADMIN.xlsb,.xlsb,20251932,5d05acde002b287d9068b5b4b8dc34dc04609d1cc16cbb...,SUCCESS
3,2024-07-13_0800,Pendências-010223.xlsx,.xlsx,23053,3cca4c48eaa038cee8bd806f6cae89ca1de1e6ee0e9996...,SUCCESS
4,2024-07-13_0800,QEC_5900055119_7_55_77.csv,.csv,129525,f8a0265c96ffd7085b4e65eff75c24ba5d3439daeabbc2...,SUCCESS
5,2024-07-13_0800,QEC_5900074722_7_55_77.csv,.csv,129534,705844e05903b043b8ffcbe1f4889db5e08aea5370d378...,SUCCESS
6,2024-07-13_0800,QEC_5900083950_7_55_77.csv,.csv,129520,e9d67a12cd2f3754ab7e647f0e330ff373509e7fe74643...,SUCCESS
7,2024-07-13_0800,QEC_5900086165_7_55_77.csv,.csv,129512,acf1d09ec8d814ce474f28a371b3b2d885d5c183130b9c...,SUCCESS
8,2024-07-13_0800,QEC_5900086169_7_55_77.csv,.csv,129519,5978cdd5de1fa7b267ff214b6e0e4f56b3b310810fdf9b...,SUCCESS
9,2024-07-13_0800,AnaliticoProjeto.csv,.csv,67337,7105eae169d8b48a7691d29b708f5ad2e7a530922a7efd...,SUCCESS


5. Validar objetos no MinIO

In [10]:
s3_client = boto3.client(
    "s3",
    endpoint_url=Settings.MINIO_ENDPOINT,
    aws_access_key_id=Settings.AWS_ACCESS_KEY,
    aws_secret_access_key=Settings.AWS_SECRET_KEY,
    region_name="us-east-1",
)

response = s3_client.list_objects_v2(
    Bucket=Settings.BUCKET_NAME,
    Prefix="landing/"
)

objects = response.get("Contents", [])

print(f"Objetos encontrados na Landing: {len(objects)}")

for obj in objects[:10]:
    print(obj["Key"])

Objetos encontrados na Landing: 36
landing/snapshot_date=2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv
landing/snapshot_date=2024-07-13_0800/CONTROLE DE MEDICOES EM ANDAMENTO/Exportação_bm_acompanhamento.xlsx
landing/snapshot_date=2024-07-13_0800/NACT/202211_ADMIN.xlsb
landing/snapshot_date=2024-07-13_0800/PENDENCIAS RDO/Pendências-010223.xlsx
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900055119_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900074722_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900083950_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900086165_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900086169_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/RELATORIO ANALITICO DO PROJETO/AnaliticoProjeto.csv


6. Sumário

In [11]:
landing_validation = {
    "total_files": len(landing_metadata_df),
    "success": int((landing_metadata_df["upload_status"] == "SUCCESS").sum()),
    "failed": int((landing_metadata_df["upload_status"] == "FAILED").sum()),
    "bucket": Settings.BUCKET_NAME,
    "prefix": "landing/",
    "status": "SUCCESS"
}

landing_validation

{'total_files': 36,
 'success': 36,
 'failed': 0,
 'bucket': 'contracts',
 'prefix': 'landing/',
 'status': 'SUCCESS'}

7. Isso libera o worker para o próximo notebook.

In [ ]:
#spark.stop()